<a href="https://colab.research.google.com/github/ladams1204/SportsBookEdge/blob/main/03_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install nba_api --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 3.5 MB/s eta 0:00:00


In [2]:
import sqlite3
import pandas as pd
import numpy as np
from google.colab import files

In [3]:
from nba_api.stats.endpoints import playergamelog
from nba_api.stats.static import players
import time

conn = sqlite3.connect('sportsbook.db')
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS nba_player_games (
    game_id TEXT,
    player_id INTEGER,
    player_name TEXT,
    game_date TEXT,
    matchup TEXT,
    wl TEXT,
    minutes INTEGER,
    points INTEGER,
    rebounds INTEGER,
    assists INTEGER,
    steals INTEGER,
    blocks INTEGER,
    turnovers INTEGER,
    fgm INTEGER,
    fga INTEGER,
    fg3m INTEGER,
    fg3a INTEGER,
    plus_minus INTEGER,
    season TEXT,
    PRIMARY KEY (game_id, player_id)
)
''')
conn.commit()

target_players = [
    "LeBron James", "Stephen Curry", "Nikola Jokic", "Luka Doncic",
    "Jayson Tatum", "Giannis Antetokounmpo", "Shai Gilgeous-Alexander",
    "Kevin Durant", "Anthony Edwards", "Joel Embiid",
    "Devin Booker", "Damian Lillard", "Jimmy Butler", "Paul George",
    "Donovan Mitchell", "Trae Young", "De'Aaron Fox", "Ja Morant",
    "Anthony Davis", "Victor Wembanyama"
]

def pull_player_season(player_name, season):
    matches = players.find_players_by_full_name(player_name)
    if not matches:
        return 0
    player = matches[0]
    log = playergamelog.PlayerGameLog(player_id=player['id'], season=season)
    df = log.get_data_frames()[0]
    if len(df) == 0:
        return 0
    df['SEASON'] = season
    df['PLAYER_NAME'] = player['full_name']
    df = df.rename(columns={
        'Game_ID': 'game_id', 'Player_ID': 'player_id',
        'PLAYER_NAME': 'player_name', 'GAME_DATE': 'game_date',
        'MATCHUP': 'matchup', 'WL': 'wl', 'MIN': 'minutes',
        'PTS': 'points', 'REB': 'rebounds', 'AST': 'assists',
        'STL': 'steals', 'BLK': 'blocks', 'TOV': 'turnovers',
        'FGM': 'fgm', 'FGA': 'fga', 'FG3M': 'fg3m', 'FG3A': 'fg3a',
        'PLUS_MINUS': 'plus_minus', 'SEASON': 'season'
    })[['game_id', 'player_id', 'player_name', 'game_date', 'matchup', 'wl',
        'minutes', 'points', 'rebounds', 'assists', 'steals', 'blocks',
        'turnovers', 'fgm', 'fga', 'fg3m', 'fg3a', 'plus_minus', 'season']]
    inserted = 0
    for _, row in df.iterrows():
        try:
            cursor.execute(f"INSERT INTO nba_player_games VALUES ({','.join(['?']*19)})", tuple(row))
            inserted += 1
        except sqlite3.IntegrityError:
            pass
    conn.commit()
    return inserted

total = 0
for season in ['2024-25', '2025-26']:
    for name in target_players:
        n = pull_player_season(name, season)
        total += n
        print(f"{season} {name}: {n} games")
        time.sleep(0.6)

print(f"\nTotal inserted: {total}")
count = cursor.execute("SELECT COUNT(*) FROM nba_player_games").fetchone()[0]
print(f"Database rows: {count}")

2024-25 LeBron James: 70 games
2024-25 Stephen Curry: 70 games
2024-25 Nikola Jokic: 70 games
2024-25 Luka Doncic: 50 games
2024-25 Jayson Tatum: 72 games
2024-25 Giannis Antetokounmpo: 67 games
2024-25 Shai Gilgeous-Alexander: 76 games
2024-25 Kevin Durant: 62 games
2024-25 Anthony Edwards: 79 games
2024-25 Joel Embiid: 19 games
2024-25 Devin Booker: 75 games
2024-25 Damian Lillard: 58 games
2024-25 Jimmy Butler: 55 games
2024-25 Paul George: 41 games
2024-25 Donovan Mitchell: 71 games
2024-25 Trae Young: 76 games
2024-25 De'Aaron Fox: 62 games
2024-25 Ja Morant: 50 games
2024-25 Anthony Davis: 51 games
2024-25 Victor Wembanyama: 46 games
2025-26 LeBron James: 60 games
2025-26 Stephen Curry: 43 games
2025-26 Nikola Jokic: 65 games
2025-26 Luka Doncic: 64 games
2025-26 Jayson Tatum: 16 games
2025-26 Giannis Antetokounmpo: 36 games
2025-26 Shai Gilgeous-Alexander: 68 games
2025-26 Kevin Durant: 78 games
2025-26 Anthony Edwards: 61 games
2025-26 Joel Embiid: 38 games
2025-26 Devin Booker